In [4]:
import pandas as pd
from google.cloud import bigquery
import glob
import os

In [5]:
# 1. CONFIGURAÇÕES DE ACESSO
# Coloque o caminho para o arquivo JSON que você baixou do Google Cloud
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "theme-park-queue-data-f2e1d4785d38.json"

In [6]:
client = bigquery.Client()
# ATENÇÃO: Usei o nome exato da tabela que você criou no passo anterior
table_id = "theme-park-queue-data.theme_park_queues.historical-data"

In [7]:
# 2. MAPEAMENTO DE LANDS (Ride ID -> Nome da Land)
bcw_land_rideid = {
    'Vila Germânica': [11329, 11366, 11332],
    'Ilha dos Piratas': [11340],
    'TriplikLand': [11330, 11328, 11373, 11326],
    'Cowboyland': [13872, 11368, 11367, 11444],
    'Mundo Animal': [11358],
    'Nerf Mania': [12325, 12326],
    'Aventura Radical': [11327, 11335, 11336],
    'Madagascar': [11338],
    'Terra da Fantasia': [11344],
    'Hot Wheels': [11459, 11334, 15407]
}

# Invertendo o dicionário para facilitar a busca por ID
id_to_land = {ride_id: land for land, ids in bcw_land_rideid.items() for ride_id in ids}

In [11]:
def realizar_carga_inteligente():
    arquivos = glob.glob('consolidado-bcw-*.csv')
    
    # 1. Buscar o que já existe no banco para evitar duplicados
    print("Consultando dados já existentes no BigQuery para evitar duplicidade...")
    query = f"SELECT DISTINCT timestamp_utc, ride_id FROM `{table_id}` WHERE park_id = 319"
    df_existente = client.query(query).to_dataframe()
    
    # Criar uma chave única para comparação rápida
    if not df_existente.empty:
        df_existente['chave'] = df_existente['timestamp_utc'].dt.strftime('%Y-%m-%d %H:%M:%S') + "_" + df_existente['ride_id'].astype(str)
        chaves_no_banco = set(df_existente['chave'])
    else:
        chaves_no_banco = set()

    for arquivo in arquivos:
        print(f"\n--- Processando: {arquivo} ---")
        df = pd.read_csv(arquivo)

        # TRATAMENTO DE FORMATO DE HORA (Lida com 13:25 e 13:25:00)
        df['datetime_temp'] = pd.to_datetime(df['data_local'] + ' ' + df['hora_local'], errors='coerce')
        
        # Ajuste UTC
        df['timestamp_utc'] = df['datetime_temp'] + pd.Timedelta(hours=3)

        # TRATAMENTO DE ID (Caso venha como 11332.0)
        df['ride_id'] = df['id'].fillna(0).astype(float).astype(int)
        
        df['park_id'] = 319
        df['park_name'] = "Beto Carrero World"
        df['timezone'] = "America/Sao_Paulo"
        df['land'] = df['ride_id'].apply(lambda x: id_to_land.get(x, "Outros"))
        df = df.rename(columns={'name': 'ride_name'})

        colunas_finais = [
            'park_id', 'park_name', 'land', 'ride_id', 
            'ride_name', 'wait_time', 'is_open', 'timestamp_utc', 'timezone'
        ]
        df_final = df[colunas_finais].dropna(subset=['timestamp_utc'])

        # DEDUPLICAÇÃO LOCAL: Comparar o que está no CSV com o que já está no Banco
        df_final['chave_local'] = df_final['timestamp_utc'].dt.strftime('%Y-%m-%d %H:%M:%S') + "_" + df_final['ride_id'].astype(str)
        
        df_para_subir = df_final[~df_final['chave_local'].isin(chaves_no_banco)].copy()
        df_para_subir = df_para_subir.drop(columns=['chave_local'])

        if df_para_subir.empty:
            print(f"⏩ Pulando {arquivo}: Todos os dados já existem no BigQuery.")
            continue

        # CARGA
        job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
        print(f"Enviando {len(df_para_subir)} novas linhas...")
        job = client.load_table_from_dataframe(df_para_subir, table_id, job_config=job_config)
        job.result() 
        
        print(f"✅ {len(df_para_subir)} linhas novas carregadas!")

In [12]:
if __name__ == "__main__":
    realizar_carga_inteligente()

Consultando dados já existentes no BigQuery para evitar duplicidade...


C:\Users\bairru\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



--- Processando: consolidado-bcw-2023.csv ---
⏩ Pulando consolidado-bcw-2023.csv: Todos os dados já existem no BigQuery.

--- Processando: consolidado-bcw-2024.csv ---
⏩ Pulando consolidado-bcw-2024.csv: Todos os dados já existem no BigQuery.

--- Processando: consolidado-bcw-2025.csv ---
Enviando 581069 novas linhas...
✅ 581069 linhas novas carregadas!

--- Processando: consolidado-bcw-2026.csv ---
Enviando 197417 novas linhas...
✅ 197417 linhas novas carregadas!
